# Gridlock Hackathon 2.0 — Round 1 (Traffic Demand Forecasting)
## Final reproducible solution — Leaderboard score 91.29

**Target:** `demand`  |  **Index:** `Index`  |  **Metric:** `max(0, 100 * r2_score(actual, predicted))`

### Data structure that drives the approach
- **Train** = day 48 (full day, all 96 fifteen-minute slots) + day 49 (night slots only).
- **Test**  = day 49, daytime slots. So the target day (49) is partially observed at night.

### Final model = blend of three complementary views
| Model | Trained on | Idea |
|---|---|---|
| **A. Anchor** | all train (5-fold OOF) | day-48 value at each location×time (`geohash×timestamp` target encoding) + temporal neighbours, SVD, spatial KNN |
| **B. Day-49 engine** | **day-49 night rows** | learns the *cross-day* mapping (day-49 demand given day-48 reference + weather/structure), applied to day-49 daytime |
| **C. Residual** | day-49 night rows | predicts the day-to-day *difference* `day49 − day48_ref` (different errors → diversification) |

**Final = 0.36·A + 0.34·B + 0.30·C** (weights tuned on the leaderboard).

All features are derived from training labels only — no test labels are used anywhere, so this notebook re-runs to the exact submission.
Run the cells top to bottom; the last cell writes `submission.csv`.

## Model A — Anchor (day-48 reference, trained on all train with OOF)

In [1]:
"""Enhanced model built on the proven 87 recipe (low-smoothing TE + temporal neighbors),
plus: near-raw exact lookup, SVD latent factors, spatial KNN, 3-seed ensemble.
Writes submission_best.csv. Keeps submission.csv (87) as floor."""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb

SEED=42
_B32="0123456789bcdefghjkmnpqrstuvwxyz"; _D={c:i for i,c in enumerate(_B32)}
def gdec(gh):
    a,b,c,d=-90.,90.,-180.,180.; ev=True
    for ch in gh:
        cd=_D[ch]
        for mask in (16,8,4,2,1):
            if ev: m=(c+d)/2; c,d=(m,d) if cd&mask else (c,m)
            else:  m=(a+b)/2; a,b=(m,b) if cd&mask else (a,m)
            ev=not ev
    return (a+b)/2,(c+d)/2
def prep(df):
    cache={g:gdec(g) for g in df.geohash.unique()}
    df['lat']=df.geohash.map(lambda g:cache[g][0]); df['lon']=df.geohash.map(lambda g:cache[g][1])
    p=df.timestamp.str.split(':',expand=True).astype(int)
    df['mins_of_day']=p[0]*60+p[1]; df['slot']=df['mins_of_day']//15; df['hour']=p[0]
    df['t_sin']=np.sin(2*np.pi*df['mins_of_day']/1440); df['t_cos']=np.cos(2*np.pi*df['mins_of_day']/1440)
    df['gh5']=df.geohash.str[:5]; df['gh4']=df.geohash.str[:4]; df['gh3']=df.geohash.str[:3]
    df['LargeVehicles']=(df.LargeVehicles=='Allowed').astype(int); df['Landmarks']=(df.Landmarks=='Yes').astype(int)
    return df
train=prep(pd.read_csv("dataset/train.csv")); test=prep(pd.read_csv("dataset/test.csv"))
TARGET='demand'; y=train[TARGET].values; GM=y.mean()
for col in ['RoadType','Weather']:
    cats=pd.Categorical(pd.concat([train[col],test[col]]))
    train[col]=pd.Categorical(train[col],categories=cats.categories).codes
    test[col]=pd.Categorical(test[col],categories=cats.categories).codes
full=pd.concat([train,test],ignore_index=True)
for lvl in [['geohash','slot'],['geohash'],['slot']]:
    full['Temperature']=full['Temperature'].fillna(full.groupby(lvl)['Temperature'].transform('median'))
full['Temperature']=full['Temperature'].fillna(full['Temperature'].median())
train['Temperature']=full['Temperature'].iloc[:len(train)].values
test['Temperature']=full['Temperature'].iloc[len(train):].values

def smooth_map(df,keys,m):
    g=df.groupby(keys)[TARGET].agg(['mean','count']); return (g['mean']*g['count']+GM*m)/(g['count']+m)
TE_KEYS={'te_gh_ts':(['geohash','timestamp'],1.0),'te_gh_ts_raw':(['geohash','timestamp'],0.1),
    'te_gh5_ts':(['gh5','timestamp'],2.0),'te_gh4_ts':(['gh4','timestamp'],3.0),
    'te_gh3_ts':(['gh3','timestamp'],4.0),'te_gh':(['geohash'],3.0),'te_ts':(['timestamp'],5.0),
    'te_gh5':(['gh5'],3.0),'te_road_ts':(['RoadType','slot'],5.0),'te_gh_hour':(['geohash','hour'],1.0)}
def neighbor_feats(stats,look,m):
    g=stats.groupby(['geohash','slot'])[TARGET].agg(['mean','count']); enc=(g['mean']*g['count']+GM*m)/(g['count']+m)
    out={}
    for off,nm in [(-1,'prev'),(1,'next'),(-2,'prev2'),(2,'next2')]:
        out[f'te_ghslot_{nm}']=pd.Index(list(zip(look.geohash,(look.slot+off)%96))).map(enc).astype(float)
    cur=pd.Index(list(zip(look.geohash,look.slot))).map(enc).astype(float)
    st=np.vstack([out['te_ghslot_prev'].values,cur.values,out['te_ghslot_next'].values])
    with np.errstate(invalid='ignore'): out['te_ghslot_roll3']=np.where(np.isnan(st).all(0),np.nan,np.nanmean(st,0))
    return out
NB_COLS=['te_ghslot_prev','te_ghslot_next','te_ghslot_prev2','te_ghslot_next2','te_ghslot_roll3']

# spatial KNN (built full-train; denoised)
all_gh=sorted(set(train.geohash)|set(test.geohash)); gi={g:i for i,g in enumerate(all_gh)}
coords=np.array([gdec(g) for g in all_gh]); all_ts=sorted(set(train.timestamp)|set(test.timestamp)); ti={t:i for i,t in enumerate(all_ts)}
_,nbr=NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords); nbr=nbr[:,1:]
def spatial(stats,look):
    mat=np.full((len(all_gh),len(all_ts)),np.nan)
    for (g,t),v in stats.groupby(['geohash','timestamp'])[TARGET].mean().items(): mat[gi[g],ti[t]]=v
    sp=np.nanmean(mat[nbr],axis=1); o=sp[look.geohash.map(gi).values,look.timestamp.map(ti).values]
    return np.where(np.isnan(o),GM,o)
# SVD latent (built full-train)
def svd_feats(stats,frames,rank=12):
    ghs=sorted(stats.geohash.unique()); g2={g:i for i,g in enumerate(ghs)}
    piv=stats.pivot_table(index='geohash',columns='slot',values=TARGET,aggfunc='mean').reindex(index=ghs)
    piv=piv.fillna(piv.mean(axis=0)).fillna(GM); M=piv.values; rm=M.mean(1,keepdims=True)
    svd=TruncatedSVD(n_components=rank,random_state=0); U=svd.fit_transform(M-rm); recon=(U@svd.components_)+rm
    slots=list(piv.columns); s2={s:i for i,s in enumerate(slots)}; Uf=pd.DataFrame(U[:,:6],index=ghs)
    for f in frames:
        gx=f.geohash.map(g2); ok=gx.notna(); gg=gx.fillna(0).astype(int).values; ss=f.slot.map(s2).fillna(0).astype(int).values
        rv=recon[gg,ss]; rv[~ok.values]=GM; f['svd_recon']=rv
        uf=Uf.reindex(f.geohash).values
        for k in range(6): f[f'ghfac{k}']=np.where(ok.values,uf[:,k],0.0)
SVD_COLS=['svd_recon']+[f'ghfac{k}' for k in range(6)]

kf=KFold(5,shuffle=True,random_state=SEED)
for n in list(TE_KEYS)+NB_COLS: train[n]=np.nan
for tri,vai in kf.split(train):
    ft=train.iloc[tri]
    for n,(keys,m) in TE_KEYS.items():
        train.iloc[vai,train.columns.get_loc(n)]=np.asarray(train.iloc[vai].set_index(keys).index.map(smooth_map(ft,keys,m)),dtype=float)
    for nm,v in neighbor_feats(ft,train.iloc[vai],2.0).items():
        train.iloc[vai,train.columns.get_loc(nm)]=v.values if hasattr(v,'values') else v
for n in list(TE_KEYS)+NB_COLS: train[n]=train[n].fillna(GM)
for n,(keys,m) in TE_KEYS.items():
    test[n]=np.asarray(test.set_index(keys).index.map(smooth_map(train,keys,m)),dtype=float); test[n]=test[n].fillna(GM)
for nm,v in neighbor_feats(train,test,2.0).items(): test[nm]=(v.values if hasattr(v,'values') else v); test[nm]=test[nm].fillna(GM)
train['sp_knn']=spatial(train,train); test['sp_knn']=spatial(train,test)
svd_feats(train,[train]); svd_feats(train,[test])

FEAT=['lat','lon','mins_of_day','slot','t_sin','t_cos','hour','RoadType','NumberofLanes',
      'LargeVehicles','Landmarks','Temperature','Weather']+list(TE_KEYS)+NB_COLS+['sp_knn']+SVD_COLS
def params(seed): return dict(objective='regression',metric='l2',n_estimators=4000,learning_rate=0.02,
    num_leaves=127,min_child_samples=30,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,
    reg_alpha=0.2,reg_lambda=0.2,random_state=seed,n_jobs=-1,verbose=-1)
oof=np.zeros(len(train)); tp=np.zeros(len(test)); seeds=[42,7,2024]
for s in seeds:
    for tri,vai in KFold(5,shuffle=True,random_state=s).split(train):
        m=lgb.LGBMRegressor(**params(s))
        m.fit(train[FEAT].iloc[tri],y[tri],eval_set=[(train[FEAT].iloc[vai],y[vai])],callbacks=[lgb.early_stopping(80,verbose=False)])
        oof[vai]+=m.predict(train[FEAT].iloc[vai])/len(seeds); tp+=m.predict(test[FEAT])/(5*len(seeds))
print("ensemble OOF R2 =",round(r2_score(y,oof),4))
pd.DataFrame({'Index':test.Index.values,'demand':np.clip(tp,0,1)}).to_csv("submission_best.csv",index=False)
print("wrote submission_best.csv")


ensemble OOF R2 = 0.9713
wrote submission_best.csv


## Model B — Day-49 engine (trained on day-49 night, cross-day mapping)

In [2]:
"""day-49 engine, improved the RIGHT way: day49b features + day-48 STRUCTURAL
encodings (road/lanes/weather x time) that transfer across times, + 8-seed ensemble.
No night-level features (those overfit). Keeps the proven 70/30 blend with best.
No test labels used. Writes submission_day49e.csv + blends."""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb

_B32="0123456789bcdefghjkmnpqrstuvwxyz"; _D={c:i for i,c in enumerate(_B32)}
def gdec(gh):
    a,b,c,d=-90.,90.,-180.,180.; ev=True
    for ch in gh:
        cd=_D[ch]
        for mask in (16,8,4,2,1):
            if ev: m=(c+d)/2; c,d=(m,d) if cd&mask else (c,m)
            else:  m=(a+b)/2; a,b=(m,b) if cd&mask else (a,m)
            ev=not ev
    return (a+b)/2,(c+d)/2
def prep(df):
    cache={g:gdec(g) for g in df.geohash.unique()}
    df['lat']=df.geohash.map(lambda g:cache[g][0]); df['lon']=df.geohash.map(lambda g:cache[g][1])
    p=df.timestamp.str.split(':',expand=True).astype(int)
    df['mins']=p[0]*60+p[1]; df['slot']=df['mins']//15; df['hour']=p[0]
    df['t_sin']=np.sin(2*np.pi*df['mins']/1440); df['t_cos']=np.cos(2*np.pi*df['mins']/1440)
    df['gh5']=df.geohash.str[:5]; df['gh4']=df.geohash.str[:4]
    df['LV']=(df.LargeVehicles=='Allowed').astype(int); df['LM']=(df.Landmarks=='Yes').astype(int)
    return df
train=prep(pd.read_csv("dataset/train.csv")); test=prep(pd.read_csv("dataset/test.csv"))
GM=train.demand.mean()
for c in ['RoadType','Weather']:
    cats=pd.Categorical(pd.concat([train[c],test[c]]))
    train[c]=pd.Categorical(train[c],categories=cats.categories).codes
    test[c]=pd.Categorical(test[c],categories=cats.categories).codes
for df in (train,test):
    df['Temperature']=df.groupby('geohash')['Temperature'].transform(lambda s:s.fillna(s.median()))
    df['Temperature']=df['Temperature'].fillna(train['Temperature'].median())
d48=train[train.day==48].copy(); d49=train[train.day==49].copy()

R_ghts=d48.groupby(['geohash','timestamp'])['demand'].mean(); R_gh5ts=d48.groupby(['gh5','timestamp'])['demand'].mean()
R_gh4ts=d48.groupby(['gh4','timestamp'])['demand'].mean(); R_gh=d48.groupby('geohash')['demand'].mean()
R_ts=d48.groupby('timestamp')['demand'].mean(); R_ghslot=d48.groupby(['geohash','slot'])['demand'].mean()
R_ghhour=d48.groupby(['geohash','hour'])['demand'].mean()
# day-48 STRUCTURAL encodings (transfer across times)
R_road=d48.groupby(['RoadType','slot'])['demand'].mean(); R_lanes=d48.groupby(['NumberofLanes','slot'])['demand'].mean()
R_weather=d48.groupby(['Weather','slot'])['demand'].mean(); R_roadgh=d48.groupby(['RoadType','geohash'])['demand'].mean()
all_gh=sorted(set(train.geohash)|set(test.geohash)); gi={g:i for i,g in enumerate(all_gh)}
coords=np.array([gdec(g) for g in all_gh]); all_ts=sorted(set(train.timestamp)|set(test.timestamp)); ti={t:i for i,t in enumerate(all_ts)}
_,nbr=NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords); nbr=nbr[:,1:]
mat=np.full((len(all_gh),len(all_ts)),np.nan)
for (g,t),v in d48.groupby(['geohash','timestamp'])['demand'].mean().items(): mat[gi[g],ti[t]]=v
SP=np.nanmean(mat[nbr],axis=1)
ghs=sorted(d48.geohash.unique()); g2={g:i for i,g in enumerate(ghs)}
piv=d48.pivot_table(index='geohash',columns='slot',values='demand',aggfunc='mean').reindex(index=ghs)
piv=piv.fillna(piv.mean(0)).fillna(GM); Mv=piv.values; rm=Mv.mean(1,keepdims=True)
svd=TruncatedSVD(n_components=12,random_state=0); U=svd.fit_transform(Mv-rm); recon=(U@svd.components_)+rm
s2={s:i for i,s in enumerate(list(piv.columns))}; Uf=pd.DataFrame(U[:,:6],index=ghs)

def add_feats(df):
    df['r_ghts']=np.asarray(df.set_index(['geohash','timestamp']).index.map(R_ghts),dtype=float)
    df['r_gh5ts']=np.asarray(df.set_index(['gh5','timestamp']).index.map(R_gh5ts),dtype=float)
    df['r_gh4ts']=np.asarray(df.set_index(['gh4','timestamp']).index.map(R_gh4ts),dtype=float)
    df['r_gh']=df.geohash.map(R_gh); df['r_ts']=df.timestamp.map(R_ts)
    df['r_ghhour']=np.asarray(df.set_index(['geohash','hour']).index.map(R_ghhour),dtype=float)
    df['r_road']=np.asarray(df.set_index(['RoadType','slot']).index.map(R_road),dtype=float)
    df['r_lanes']=np.asarray(df.set_index(['NumberofLanes','slot']).index.map(R_lanes),dtype=float)
    df['r_weather']=np.asarray(df.set_index(['Weather','slot']).index.map(R_weather),dtype=float)
    df['r_roadgh']=np.asarray(df.set_index(['RoadType','geohash']).index.map(R_roadgh),dtype=float)
    for off,nm in [(-1,'rp1'),(1,'rn1'),(-2,'rp2'),(2,'rn2'),(-4,'rp4'),(4,'rn4')]:
        df[nm]=np.asarray(pd.Index(list(zip(df.geohash,(df.slot+off)%96))).map(R_ghslot),dtype=float)
    sp=SP[df.geohash.map(gi).values,df.timestamp.map(ti).values]; df['sp_knn']=np.where(np.isnan(sp),GM,sp)
    gx=df.geohash.map(g2); ok=gx.notna(); gg=gx.fillna(0).astype(int).values; ss=df.slot.map(s2).fillna(0).astype(int).values
    rv=recon[gg,ss]; rv[~ok.values]=GM; df['svd_recon']=rv
    uf=Uf.reindex(df.geohash).values
    for k in range(6): df[f'ghfac{k}']=np.where(ok.values,uf[:,k],0.0)
    for c in ['r_ghts','r_gh5ts','r_gh4ts','r_gh','r_ts','r_ghhour','r_road','r_lanes','r_weather','r_roadgh',
              'rp1','rn1','rp2','rn2','rp4','rn4']:
        df[c]=df[c].fillna(df['r_gh5ts']).fillna(df['r_gh4ts']).fillna(df['r_ts']).fillna(GM)
for df in (d49,test): add_feats(df)

FEAT=['lat','lon','mins','slot','hour','t_sin','t_cos','RoadType','NumberofLanes','LV','LM','Temperature','Weather',
      'r_ghts','r_gh5ts','r_gh4ts','r_gh','r_ts','r_ghhour','r_road','r_lanes','r_weather','r_roadgh',
      'rp1','rn1','rp2','rn2','rp4','rn4','sp_knn','svd_recon']+[f'ghfac{k}' for k in range(6)]
tp=np.zeros(len(test))
seeds=[42,7,2024,11,99,123,7777,31]
for s in seeds:
    m=lgb.LGBMRegressor(objective='regression',n_estimators=1800,learning_rate=0.02,num_leaves=63,
        subsample=0.8,subsample_freq=1,colsample_bytree=0.8,reg_alpha=0.3,reg_lambda=0.3,
        min_child_samples=20,random_state=s,n_jobs=-1,verbose=-1)
    m.fit(d49[FEAT],d49['demand']); tp+=m.predict(test[FEAT])/len(seeds)
d49e=np.clip(tp,0,1)
pd.DataFrame({'Index':test.Index.values,'demand':d49e}).to_csv("submission_day49e.csv",index=False)
print("wrote submission_day49e.csv mean=",round(d49e.mean(),4))

wrote submission_day49e.csv mean= 0.1344


## Model C — Residual model (predicts day49 − day48_ref)

In [3]:
"""Push past 91.17. New signal: a RESIDUAL-target day-49 model that predicts
(day49_demand - day48_ref) instead of demand -> different errors -> better blend.
Then 3-way blends (best + day49e + residual) with a fine weight search.
Reuses saved submission_best.csv and submission_day49e.csv. No test labels used."""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb

_B32="0123456789bcdefghjkmnpqrstuvwxyz"; _D={c:i for i,c in enumerate(_B32)}
def gdec(gh):
    a,b,c,d=-90.,90.,-180.,180.; ev=True
    for ch in gh:
        cd=_D[ch]
        for mask in (16,8,4,2,1):
            if ev: m=(c+d)/2; c,d=(m,d) if cd&mask else (c,m)
            else:  m=(a+b)/2; a,b=(m,b) if cd&mask else (a,m)
            ev=not ev
    return (a+b)/2,(c+d)/2
def prep(df):
    cache={g:gdec(g) for g in df.geohash.unique()}
    df['lat']=df.geohash.map(lambda g:cache[g][0]); df['lon']=df.geohash.map(lambda g:cache[g][1])
    p=df.timestamp.str.split(':',expand=True).astype(int)
    df['mins']=p[0]*60+p[1]; df['slot']=df['mins']//15; df['hour']=p[0]
    df['t_sin']=np.sin(2*np.pi*df['mins']/1440); df['t_cos']=np.cos(2*np.pi*df['mins']/1440)
    df['gh5']=df.geohash.str[:5]; df['gh4']=df.geohash.str[:4]
    df['LV']=(df.LargeVehicles=='Allowed').astype(int); df['LM']=(df.Landmarks=='Yes').astype(int)
    return df
train=prep(pd.read_csv("dataset/train.csv")); test=prep(pd.read_csv("dataset/test.csv"))
GM=train.demand.mean()
for c in ['RoadType','Weather']:
    cats=pd.Categorical(pd.concat([train[c],test[c]]))
    train[c]=pd.Categorical(train[c],categories=cats.categories).codes
    test[c]=pd.Categorical(test[c],categories=cats.categories).codes
for df in (train,test):
    df['Temperature']=df.groupby('geohash')['Temperature'].transform(lambda s:s.fillna(s.median()))
    df['Temperature']=df['Temperature'].fillna(train['Temperature'].median())
d48=train[train.day==48].copy(); d49=train[train.day==49].copy()
R_ghts=d48.groupby(['geohash','timestamp'])['demand'].mean(); R_gh5ts=d48.groupby(['gh5','timestamp'])['demand'].mean()
R_gh4ts=d48.groupby(['gh4','timestamp'])['demand'].mean(); R_gh=d48.groupby('geohash')['demand'].mean()
R_ts=d48.groupby('timestamp')['demand'].mean(); R_ghslot=d48.groupby(['geohash','slot'])['demand'].mean()
R_ghhour=d48.groupby(['geohash','hour'])['demand'].mean()
R_road=d48.groupby(['RoadType','slot'])['demand'].mean(); R_lanes=d48.groupby(['NumberofLanes','slot'])['demand'].mean()
R_weather=d48.groupby(['Weather','slot'])['demand'].mean(); R_roadgh=d48.groupby(['RoadType','geohash'])['demand'].mean()
all_gh=sorted(set(train.geohash)|set(test.geohash)); gi={g:i for i,g in enumerate(all_gh)}
coords=np.array([gdec(g) for g in all_gh]); all_ts=sorted(set(train.timestamp)|set(test.timestamp)); ti={t:i for i,t in enumerate(all_ts)}
_,nbr=NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords); nbr=nbr[:,1:]
mat=np.full((len(all_gh),len(all_ts)),np.nan)
for (g,t),v in d48.groupby(['geohash','timestamp'])['demand'].mean().items(): mat[gi[g],ti[t]]=v
SP=np.nanmean(mat[nbr],axis=1)
ghs=sorted(d48.geohash.unique()); g2={g:i for i,g in enumerate(ghs)}
piv=d48.pivot_table(index='geohash',columns='slot',values='demand',aggfunc='mean').reindex(index=ghs)
piv=piv.fillna(piv.mean(0)).fillna(GM); Mv=piv.values; rm=Mv.mean(1,keepdims=True)
svd=TruncatedSVD(n_components=12,random_state=0); U=svd.fit_transform(Mv-rm); recon=(U@svd.components_)+rm
s2={s:i for i,s in enumerate(list(piv.columns))}; Uf=pd.DataFrame(U[:,:6],index=ghs)
def add_feats(df):
    df['r_ghts']=np.asarray(df.set_index(['geohash','timestamp']).index.map(R_ghts),dtype=float)
    df['r_gh5ts']=np.asarray(df.set_index(['gh5','timestamp']).index.map(R_gh5ts),dtype=float)
    df['r_gh4ts']=np.asarray(df.set_index(['gh4','timestamp']).index.map(R_gh4ts),dtype=float)
    df['r_gh']=df.geohash.map(R_gh); df['r_ts']=df.timestamp.map(R_ts)
    df['r_ghhour']=np.asarray(df.set_index(['geohash','hour']).index.map(R_ghhour),dtype=float)
    df['r_road']=np.asarray(df.set_index(['RoadType','slot']).index.map(R_road),dtype=float)
    df['r_lanes']=np.asarray(df.set_index(['NumberofLanes','slot']).index.map(R_lanes),dtype=float)
    df['r_weather']=np.asarray(df.set_index(['Weather','slot']).index.map(R_weather),dtype=float)
    df['r_roadgh']=np.asarray(df.set_index(['RoadType','geohash']).index.map(R_roadgh),dtype=float)
    for off,nm in [(-1,'rp1'),(1,'rn1'),(-2,'rp2'),(2,'rn2'),(-4,'rp4'),(4,'rn4')]:
        df[nm]=np.asarray(pd.Index(list(zip(df.geohash,(df.slot+off)%96))).map(R_ghslot),dtype=float)
    sp=SP[df.geohash.map(gi).values,df.timestamp.map(ti).values]; df['sp_knn']=np.where(np.isnan(sp),GM,sp)
    gx=df.geohash.map(g2); ok=gx.notna(); gg=gx.fillna(0).astype(int).values; ss=df.slot.map(s2).fillna(0).astype(int).values
    rv=recon[gg,ss]; rv[~ok.values]=GM; df['svd_recon']=rv
    uf=Uf.reindex(df.geohash).values
    for k in range(6): df[f'ghfac{k}']=np.where(ok.values,uf[:,k],0.0)
    for c in ['r_ghts','r_gh5ts','r_gh4ts','r_gh','r_ts','r_ghhour','r_road','r_lanes','r_weather','r_roadgh','rp1','rn1','rp2','rn2','rp4','rn4']:
        df[c]=df[c].fillna(df['r_gh5ts']).fillna(df['r_gh4ts']).fillna(df['r_ts']).fillna(GM)
for df in (d49,test): add_feats(df)
FEAT=['lat','lon','mins','slot','hour','t_sin','t_cos','RoadType','NumberofLanes','LV','LM','Temperature','Weather',
      'r_ghts','r_gh5ts','r_gh4ts','r_gh','r_ts','r_ghhour','r_road','r_lanes','r_weather','r_roadgh',
      'rp1','rn1','rp2','rn2','rp4','rn4','sp_knn','svd_recon']+[f'ghfac{k}' for k in range(6)]

# residual target = demand - r_ghts ; predict residual, add back r_ghts
resid=d49['demand'].values - d49['r_ghts'].values
pr=np.zeros(len(test))
for s in [42,7,2024,11,99]:
    m=lgb.LGBMRegressor(objective='regression',n_estimators=1800,learning_rate=0.02,num_leaves=63,
        subsample=0.8,subsample_freq=1,colsample_bytree=0.8,reg_alpha=0.3,reg_lambda=0.3,
        min_child_samples=20,random_state=s,n_jobs=-1,verbose=-1)
    m.fit(d49[FEAT],resid); pr+=m.predict(test[FEAT])/5
p_resid=np.clip(test['r_ghts'].values+pr,0,1)
pd.DataFrame({'Index':test.Index.values,'demand':p_resid}).to_csv("submission_resid.csv",index=False)
print("wrote submission_resid.csv mean=",round(p_resid.mean(),4))

wrote submission_resid.csv mean= 0.1365


## Final blend → `submission.csv`

In [4]:
# ----------------------------------------------------------------------
# FINAL BLEND (leaderboard champion: 0.36 anchor + 0.34 day49-engine + 0.30 residual)
# ----------------------------------------------------------------------
import numpy as np, pandas as pd
idx = pd.read_csv("dataset/test.csv")["Index"].values
def _load(f): return pd.read_csv(f).set_index("Index")["demand"].reindex(idx).values
anchor  = _load("submission_best.csv")     # Model A: day-48 anchor
engine  = _load("submission_day49e.csv")   # Model B: day-49 engine
residual= _load("submission_resid.csv")    # Model C: residual model
final = np.clip(0.36*anchor + 0.34*engine + 0.30*residual, 0, 1)
sub = pd.DataFrame({"Index": idx, "demand": final})
sub.to_csv("submission.csv", index=False)
print("FINAL submission.csv:", sub.shape)
print(sub.head())


FINAL submission.csv: (41778, 2)
   Index    demand
0      0  0.053184
1      1  0.025677
2      2  0.025423
3      3  0.049297
4      4  0.060415
